In [ ]:
import redis

r = redis.Redis(host="localhost", port=6379, decode_responses=True)
print("Ping:", r.ping())
print("Redis version:", r.info("server")["redis_version"])

Ping: True
Redis version: 7.4.8


---
## 2.1.1 Строки и TTL

In [ ]:
r.set("greeting", "Hello, Redis!")

value = r.get("greeting")
print(f"GET greeting -> {value}")

GET greeting -> Hello, Redis!


In [ ]:
print(f"TTL greeting -> {r.ttl('greeting')}")

r.expire("greeting", 30)

print(f"TTL greeting -> {r.ttl('greeting')} сек")

TTL greeting -> -1
TTL greeting -> 30 сек


In [ ]:
r.delete("greeting")
print(f"GET после DEL -> {r.get('greeting')}")

print(f"TTL несуществующего ключа -> {r.ttl('greeting')}")

GET после DEL -> None
TTL несуществующего ключа -> -2


---
## 2.1.2 Хэши

In [ ]:
r.hset("user:1", mapping={"name": "Иван", "age": 25, "city": "Москва"})

name = r.hget("user:1", "name")
print(f"HGET user:1 name -> {name}")

profile = r.hgetall("user:1")
print(f"HGETALL user:1 -> {profile}")

HGET user:1 name -> Иван
HGETALL user:1 -> {'name': 'Иван', 'age': '25', 'city': 'Москва'}


In [ ]:
new_age = r.hincrby("user:1", "age", 1)
print(f"HINCRBY age +1 -> age = {new_age}")

r.hdel("user:1", "city")
profile = r.hgetall("user:1")
print(f"HGETALL после HDEL city -> {profile}")

HINCRBY age +1 -> age = 26
HGETALL после HDEL city -> {'name': 'Иван', 'age': '26'}


---
## 2.1.3 Списки

In [ ]:
r.delete("posts:recent")

r.lpush("posts:recent", "Пост 1")
r.lpush("posts:recent", "Пост 2")
r.lpush("posts:recent", "Пост 3")

# LRANGE — вывести все элементы (0 = первый, -1 = последний)
posts = r.lrange("posts:recent", 0, -1)
print(f"LRANGE posts:recent 0 -1 -> {posts}")

LRANGE posts:recent 0 -1 -> ['Пост 3', 'Пост 2', 'Пост 1']


In [ ]:
# LINDEX — получить элемент по индексу без удаления
latest = r.lindex("posts:recent", 0)
print(f"LINDEX 0 (последний добавленный) -> {latest}")

# RPOP — удалить последний элемент (хвост списка)
removed = r.rpop("posts:recent")
print(f"RPOP (удалён из хвоста) -> {removed}")
print(f"Список после RPOP -> {r.lrange('posts:recent', 0, -1)}")

LINDEX 0 (последний добавленный) -> Пост 3
RPOP (удалён из хвоста) -> Пост 1
Список после RPOP -> ['Пост 3', 'Пост 2']


In [ ]:
# Добавим элементы обратно для демонстрации LTRIM
r.lpush("posts:recent", "Пост 1")
print(f"Перед LTRIM -> {r.lrange('posts:recent', 0, -1)}")

# LTRIM — обрезать список до двух элементов
r.ltrim("posts:recent", 0, 1)
print(f"После LTRIM 0 1 -> {r.lrange('posts:recent', 0, -1)}")

Перед LTRIM -> ['Пост 1', 'Пост 3', 'Пост 2']
После LTRIM 0 1 -> ['Пост 1', 'Пост 3']


---
## 2.1.4 Множества

In [ ]:
r.delete("post:101:tags")

r.sadd("post:101:tags", "redis", "database", "nosql")
r.sadd("post:101:tags", "redis", "tutorial")

# SMEMBERS — получить все теги
tags = r.smembers("post:101:tags")
print(f"SMEMBERS -> {tags}")

SMEMBERS -> {'redis', 'tutorial', 'nosql', 'database'}


In [ ]:
# SISMEMBER — проверить наличие тега
print(f'SISMEMBER "mongodb" -> {r.sismember("post:101:tags", "mongodb")}')
print(f'SISMEMBER "redis"   -> {r.sismember("post:101:tags", "redis")}')

# SREM — удалить тег
r.srem("post:101:tags", "nosql")
print(f'После SREM "nosql" -> {r.smembers("post:101:tags")}')

# SCARD — количество тегов
print(f"SCARD -> {r.scard('post:101:tags')}")

SISMEMBER "mongodb" -> 0
SISMEMBER "redis"   -> 1
После SREM "nosql" -> {'redis', 'tutorial', 'database'}
SCARD -> 3


In [ ]:
# SINTER — пересечение тегов двух постов
r.sadd("post:102:tags", "redis", "python", "tutorial")
common = r.sinter("post:101:tags", "post:102:tags")
print(f"Общие теги постов 101 и 102: {common}")

Общие теги постов 101 и 102: {'redis', 'tutorial'}


---
## 2.2 Чатик

In [ ]:
import json
import threading
import time
from datetime import datetime

import redis

CHANNEL = "RUDN_CHAT"
HISTORY_KEY = "RUDN_CHAT_HISTORY"
HISTORY_MAX = 20


def save_message(rc: redis.Redis, username: str, message: str) -> None:
    """Сохранить сообщение в историю (список, макс. 20 элементов)."""
    payload = json.dumps(
        {
            "username": username,
            "message": message,
            "timestamp": datetime.now().strftime("%H:%M:%S"),
        },
        ensure_ascii=False,
    )
    rc.lpush(HISTORY_KEY, payload)
    rc.ltrim(HISTORY_KEY, 0, HISTORY_MAX - 1)  # Храним только последние 20


def load_history(rc: redis.Redis) -> list:
    """Загрузить историю (последние 20 сообщений, хронологически)."""
    raw = rc.lrange(HISTORY_KEY, 0, HISTORY_MAX - 1)
    return [json.loads(m) for m in reversed(raw)]


def send_message(rc: redis.Redis, username: str, message: str) -> None:
    """Сохранить в историю и опубликовать в канал."""
    save_message(rc, username, message)
    payload = json.dumps(
        {
            "username": username,
            "message": message,
            "timestamp": datetime.now().strftime("%H:%M:%S"),
        },
        ensure_ascii=False,
    )
    rc.publish(CHANNEL, payload)


def receive_messages(pubsub, stop_event: threading.Event) -> None:
    """Поток: слушаем входящие сообщения из канала."""
    try:
        for raw_msg in pubsub.listen():
            if stop_event.is_set():
                break
            if raw_msg["type"] == "message":
                msg = json.loads(raw_msg["data"])
                print(f"[{msg['timestamp']}] {msg['username']}: {msg['message']}")
    except (ConnectionError, OSError, ValueError):
        pass


def run(username: str, messages: list, delay: float = 0.5) -> None:
    """
    Запуск чата.

    - Показывает историю
    - Запускает поток-подписчик
    - Отправляет список сообщений с паузой
    - Останавливает подписчик
    """
    rc = redis.Redis(host="localhost", port=6379, decode_responses=True)
    pubsub = rc.pubsub()
    pubsub.subscribe(CHANNEL)

    # история
    history = load_history(rc)
    if history:
        print("=== История чата ===")
        for m in history:
            print(f"  [{m['timestamp']}] {m['username']}: {m['message']}")
        print("===================")
    else:
        print("История пуста.")

    # поток приёма
    stop_event = threading.Event()
    listener = threading.Thread(
        target=receive_messages, args=(pubsub, stop_event), daemon=True
    )
    listener.start()

    # отправка
    for msg in messages:
        time.sleep(delay)
        print(f">>> Отправка: {msg}")
        send_message(rc, username, msg)

    time.sleep(0.5)  # дать подписчику время принять последнее сообщение
    stop_event.set()
    pubsub.unsubscribe(CHANNEL)
    pubsub.close()
    print("\n[Сеанс завершён]")

In [ ]:
r.delete(HISTORY_KEY)

run(
    username="Алиса",
    messages=["Привет всем!", "Redis Pub/Sub работает отлично", "До свидания!"],
)

История пуста.
>>> Отправка: Привет всем!
[14:09:31] Алиса: Привет всем!
>>> Отправка: Redis Pub/Sub работает отлично
[14:09:32] Алиса: Redis Pub/Sub работает отлично
>>> Отправка: До свидания!
[14:09:32] Алиса: До свидания!

[Сеанс завершён]


In [ ]:
run(username="Боб", messages=["Привет, Алиса! Я тоже здесь"])

=== История чата ===
  [14:09:31] Алиса: Привет всем!
  [14:09:32] Алиса: Redis Pub/Sub работает отлично
  [14:09:32] Алиса: До свидания!
>>> Отправка: Привет, Алиса! Я тоже здесь
[14:09:33] Боб: Привет, Алиса! Я тоже здесь

[Сеанс завершён]


In [ ]:
print("=== Финальная история ===")
for m in load_history(r):
    print(f"  [{m['timestamp']}] {m['username']}: {m['message']}")

=== Финальная история ===
  [14:09:31] Алиса: Привет всем!
  [14:09:32] Алиса: Redis Pub/Sub работает отлично
  [14:09:32] Алиса: До свидания!
  [14:09:33] Боб: Привет, Алиса! Я тоже здесь
